## 机器学习之判别式算法和生成式算法 （2024年3月13日记）



**前言** 前几日跟着Andrew Ng的CS229学习了线性回归,逻辑回归,广义线性模型，感知机，朴素贝叶斯，高斯判别算法（GDA),支持向量机等，看了一下李航的《统计学习方法》，但是总感觉学了之后脑子里还是有一点点糊涂，感觉刷的有点太快了，并没有完全的全记住，所以想借这个博客能够帮助自己更好地回顾一下这些内容。这篇文章将主要讲解判别式算法和生成式算法的分类，来帮助我们从整体上，对这些算法设计的目的有一个更好的理解。

在开始前，我先卖一个关子。首先，毋庸置疑的是，我们大部分人第一次接触机器学习都是从线性回归开始入门的，那么我们现在回过头来再看看什么线性回归。用简单的话来说（以房价预测为例吧），线性回归就是，有n组特征分别是$X_1,X_2...X_n$，这些都是会影响房价$Y$的n个指标。现在我们假设有m组样本数据（注意！！一定要把m和n两个分清楚了，n是特征的数量，m是样本的数量，不能搞混掉），我们就可以根据这m组样本来拟合线性函数$ Y=\beta_0+\beta_1X_1+...+\beta_nX_n$。那么具体是怎么去做到的呢？想一下如果现在拟合了一条不咋样的直线，那肯定有很多点是偏离这个直线的嘛，所以我们就要有一个办法，去衡量这个偏离程度，怎么样才算拟合的好？如果不好的话，我们又怎么去把这条很烂的直线（就是如图中我画的那个红线）调整到它该在的位置（绿线）呢？  

<img src="pic1.png" width = "500" height = "500" alt="图片名称" align=center />

很多人大概已经想到了，根据拟合出来的$\hat{y}$(这个是我们后面根据X和拟合直线得到的一个拟合值），直接去看训练集中$\hat{y}$和真实的y差距多少不就好了？为了防止产生负值，给它加个平方就好了。也就是**平方和误差（SSE）**，$\sum_{i=1}^n (\hat{y_i}-y_i)^2$，这个就是残差平方和，另外还有均方误差（MSE），这个是$\frac{1}{n}\sum_{i=1}^n (\hat{y_i}-y_i)^2$。

**但是**大家不知道有没有什么疑惑？反正我当时就脑子里面有两个问题，第一个问题是为什么是平方？难道不能是四次方吗？第二个问题是如果我用各个真实点到直线的距离之和来当损失函数会怎么样？

首先第一个问题，我在看了CS229中Andrew Ng的讲解之后，初步有了一些理解。他是怎么证明为什么是二次方的呢？在原视频里，他给了一个高斯噪声的假设，由最大似然可以推导出最小二乘。（这里贴一张我的听课笔记，不懂的话也可以去看原视频,在局部加权回归那个视频里面提到的）  

<img src="pic2.png" width = "700" height = "700" alt="图片名称" align=center />

还有一个问题就是为什么没用到直线距离平方之和？我个人觉得应该是以下原因：平方损失函数的凸性意味着任何局部最小值也是全局最小值，这使得找到最优解变得相对简单。相比之下，如果使用直线的垂直距离平方和作为损失函数，问题可能会变得非凸，导致优化过程更复杂，可能存在多个局部最小值。（ps:其实我当时还在知乎上看到一篇文章说因为这是最符合自然思想的，也有认为不一定这种方法是最好的，看实际情况，有的时候用绝对值误差也行，帖个传送门（https://www.zhihu.com/question/2409502 ）反正这个问题仁者见仁，智者见智吧。。至于怎么最小化这个损失函数，可以采用梯度下降法，对于线性函数的平方损失函数来说，梯度下降法是一定只有一个最小值的，不会存在局部最小的情况，这里不再过多讨论了。

好了好了，跑偏了主题，我们回到原先的话题，我之前为什么要引入一个线性回归呢？请大家在脑子里面仔细想一下，刚刚这个过程是不是我们直接目标结果去建模的一个过程？我们先得到一个结果，再对这个结果建模一步步得通过现有的训练数据和损失函数来达到优化的一个目的。这个过程就是我想说的一种判别式模型的一个过程。

现在来说一说什么是判别式模型和生成式模型。具体来说，判别式模型直接建模条件概率$P(Y|X)$,即给定输入特征$X$的情况下输出标签$Y$的条件概率。这种模型不试图理解数据生成的过程，而是专注于找出不同类别之间的边界或者区别。之前我一直挺迷惑为什么线性回归是属于判别式模型的，但是后面一想，根据损失函数修正参数的过程就是对已有的后验概率分布去进行修正，优化的过程，是直接对结果建模的。这一切也都能解释得通了。
而生成式模型呢，会对每一种类别都去建立一个模型，假设我们已经有了一些数据集，比如说一部分数据表示这些类别房子是属于中式建筑，另一部分表示这些类别房子属于西式建筑，还有一部分建筑特征表示这部分建筑属于哥布林小屋，好了，我们就对这三种类别的数据各自建模，你可以理解为形成三个簇。然后现在来了一个房子，我们不知道它是哪种房子，我们就要根据它的各种特征，把它的特征集代入到各种建筑类别的特征集建立的模型中去，去判断它是哪种房子，如果它的特征是有茅草屋顶，小小的，那么这个建筑属于哥布林小屋的概率就更大，而在其他两个模型中输出的概率就更小，所以我们认为它是哥布林小屋。现在再配上公式看一下： $P(Y|X)=\frac{P(X,Y)}{P(X)}$,这是什么意思呢？这是计算新样本跟三个类别的联合概率，用贝叶斯公式得到的。下面的$P(X)$可以展开得到$\sum_{y_i\in Y}P(X=x|Y=y_i)*P(Y=y_i)$，这里稍微解释一下，$y_i$是目标类别空间$Y$的元素，比如刚刚就是三种。X=x是指特征等于某个x特征矩阵时，$P(Y=y_i)$是指各个类别的概率这个好理解，那$P(X=x|Y=y_i)$呢？我的解释是，比如说我们已经知道这个房子是哥布林小屋，但是我们还不知道这个房子的特征，那这个房子的特征X=x的概率是多少（是哥布林小屋的话，那房子屋顶特征是茅草的概率会更大）。我们假设X这个特征集中，所有的特征都是独立分布的，那么原式子就可以改写为：$\sum_{y_i\in Y}P(Y=y_i)\prod_{j}P(X=x_j|Y=y_i)$，这里面的$x_j$就是各种特征。因为假设是独立的，所以可以连乘在一起。其实这个就是朴素贝叶斯的原理，



-------------------分割线




